# TFM – Fase 5 BI

**Notebook de verificación del entorno Python (Conda/Jupyter) – Ubuntu 24.04**  
Generado: `2026-01-22 21:24:49`

Este notebook sirve como **evidencia reproducible**: valida que el entorno activo tiene las librerías y versiones definidas en `librerias.txt`, y deja un resumen de versiones para adjuntar en el informe.


## 1) Contexto y propósito

- **Archivo de referencia:** `librerias.txt` (pip requirements)
- **Objetivo:** verificar instalación + versiones + importación correcta.

> Nota: los subrayados rojos en el editor de texto son del corrector ortográfico, no errores de Python.


## 2) Requisitos previos (terminal)

Ejecuta esto **en terminal** antes de abrir Jupyter (o dentro de un notebook con `!` si lo prefieres):

```bash
cd ~/Documents/Proyectos/TFM/docs/evidence/fase5_BI
conda env list
conda activate <TU_ENTORNO>
# Ejemplo: conda activate tfm
pip install -r librerias.txt
jupyter lab
```

Luego, en Jupyter: **Kernel → Change Kernel →** el kernel asociado al entorno.


In [1]:
# 3) Diagnóstico del intérprete y entorno activo
import sys, os, platform

print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])
print("Platform         :", platform.platform())
print("Working dir      :", os.getcwd())


Python executable: /home/engineer/anaconda3/envs/tfm/bin/python
Python version   : 3.10.19
Platform         : Linux-6.8.0-90-lowlatency-x86_64-with-glibc2.39
Working dir      : /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase5_BI


In [2]:
# 4) Leer requerimientos desde 'librerias.txt'
from pathlib import Path

req_path = Path("librerias.txt")
assert req_path.exists(), f"No se encontró {req_path.resolve()}. Ejecuta el notebook desde la carpeta correcta."

reqs = []
for line in req_path.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    reqs.append(line)

print("Requerimientos detectados:")
for r in reqs:
    print(" -", r)


Requerimientos detectados:
 - scikit-learn==1.2.1
 - pandas==1.5.1
 - matplotlib==3.7.0
 - seaborn==0.12.2
 - numpy==1.23.5
 - statsmodels==0.13.5
 - scipy==1.10.0
 - openpyxl==3.1.2


In [3]:
# 5) Verificar importaciones y versiones instaladas vs requeridas
import importlib
import re
from importlib import metadata

def norm_pkg_name(name: str) -> str:
    return name.replace("_","-").lower()

def parse_req(req: str):
    m = re.match(r"^\s*([A-Za-z0-9_\-]+)\s*==\s*([^\s]+)\s*$", req)
    if not m:
        return req.strip(), None
    return m.group(1).strip(), m.group(2).strip()

# Import name != distribution name (casos típicos)
import_name_map = {
    "scikit-learn": "sklearn",
    "statsmodels": "statsmodels",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "openpyxl": "openpyxl",
}

rows = []
for r in reqs:
    pkg, expected = parse_req(r)
    dist = pkg
    dist_norm = norm_pkg_name(dist)

    installed = None
    try:
        installed = metadata.version(dist)
    except metadata.PackageNotFoundError:
        # fallback común: sklearn -> scikit-learn
        if dist_norm == "sklearn":
            try:
                installed = metadata.version("scikit-learn")
                dist = "scikit-learn"
            except metadata.PackageNotFoundError:
                installed = None

    ok_ver = (installed is not None) if expected is None else (installed == expected)

    imp_name = import_name_map.get(dist_norm, dist_norm)
    ok_imp = True
    try:
        importlib.import_module(imp_name)
    except Exception:
        ok_imp = False

    rows.append((pkg, expected, installed, ok_ver, ok_imp, imp_name))

print(f"{'Paquete':<15} {'Req':<10} {'Instalado':<10} {'OK_ver':<7} {'OK_import':<9} ImportName")
for pkg, exp, inst, okv, okimp, impname in rows:
    print(f"{pkg:<15} {str(exp):<10} {str(inst):<10} {str(okv):<7} {str(okimp):<9} {impname}")

missing = [r for r in rows if r[2] is None]
badver  = [r for r in rows if (r[1] is not None and r[2] is not None and r[2] != r[1])]
badimp  = [r for r in rows if r[4] is False]

print("\nResumen:")
print(" - Faltantes   :", len(missing))
print(" - Versión dif :", len(badver))
print(" - Import falla:", len(badimp))

if missing or badver or badimp:
    raise RuntimeError("El entorno NO cumple completamente con librerias.txt. Revisa el resumen anterior.")
else:
    print("✅ Entorno OK: coincide con librerias.txt y las importaciones funcionan.")


Paquete         Req        Instalado  OK_ver  OK_import ImportName
scikit-learn    1.2.1      1.2.1      True    True      sklearn
pandas          1.5.1      1.5.1      True    True      pandas
matplotlib      3.7.0      3.7.0      True    True      matplotlib
seaborn         0.12.2     0.12.2     True    True      seaborn
numpy           1.23.5     1.23.5     True    True      numpy
statsmodels     0.13.5     0.13.5     True    True      statsmodels
scipy           1.10.0     1.10.0     True    True      scipy
openpyxl        3.1.2      3.1.2      True    True      openpyxl

Resumen:
 - Faltantes   : 0
 - Versión dif : 0
 - Import falla: 0
✅ Entorno OK: coincide con librerias.txt y las importaciones funcionan.


In [4]:
# 6) Guardar evidencia en un .txt en la misma carpeta (lado de librerias.txt)
from pathlib import Path
import datetime

out = Path("evidencia_versions.txt")

lines = []
lines.append("TFM – Fase5_BI – Evidencia de entorno")
lines.append(f"Generado: {datetime.datetime.now().isoformat(timespec='seconds')}")
lines.append("")
lines.append(f"Python executable: {sys.executable}")
lines.append(f"Python version   : {sys.version.split()[0]}")
lines.append("")
lines.append(f"{'Paquete':<15} {'Req':<10} {'Instalado':<10} {'OK_ver':<7} {'OK_import':<9} ImportName")
for pkg, exp, inst, okv, okimp, impname in rows:
    lines.append(f"{pkg:<15} {str(exp):<10} {str(inst):<10} {str(okv):<7} {str(okimp):<9} {impname}")

out.write_text("\n".join(lines), encoding="utf-8")
print("Evidencia guardada en:", out.resolve())


Evidencia guardada en: /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase5_BI/evidencia_versions.txt


## 7) (Opcional) Registrar kernel del entorno

Si quieres que el kernel aparezca en Jupyter con nombre claro:

```bash
conda activate <TU_ENTORNO>
python -m ipykernel install --user --name tfm --display-name "Python (TFM)"
```

Luego en Jupyter: **Kernel → Change Kernel → Python (TFM)**
